# Build Delft3D submodels in parallel (D-HyDAMO)

Builds one Delft3D/DFM model per submodel under
`output_2_cleaned/submodels/submodel_XX/`, **in parallel**.

Each build runs in its **own subprocess** via `tool/scripts/build_submodel.py`
(one submodel per process). Separate processes are required because the build
applies global monkeypatches (`StorageNodes.add_storagenode`,
`Link1d2d.link1d2d_contact_type`) that cannot be shared safely across threads;
each subprocess re-runs them on import.

- **`MAX_WORKERS`** (config cell) caps how many build at once. Builds are
  RAM-heavy, so keep this modest. Default: 3.
- Each build's log is written to `submodels/_build_logs/<submodel>.log`.
- Inter-submodel links (`submodels/links/*.gpkg`) are **not** built here.
- Config (`build.json`, `source.yaml`) and the shared DEM are found by the
  script automatically, relative to each submodel dir.

In [ ]:
# ---- configuration ----
import sys, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# notebook runs from notebook/ ; paths are relative to the repo root
DATASET_ROOT  = Path("../houston/houston_centre/output_2_cleaned").resolve()
SUBMODELS_DIR = DATASET_ROOT / "submodels"
BUILD_SCRIPT  = Path("../tool/scripts/build_submodel.py").resolve()
LOG_DIR       = SUBMODELS_DIR / "_build_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert BUILD_SCRIPT.exists(), BUILD_SCRIPT
assert SUBMODELS_DIR.exists(), SUBMODELS_DIR

# submodels to build (default: all present). Edit to a subset, e.g. ["submodel_01"].
SUBMODELS = sorted(p.name for p in SUBMODELS_DIR.glob("submodel_*") if p.is_dir())

# >>> how many submodels to build at once (RAM-bound). Change this freely. <<<
MAX_WORKERS = 3

PYTHON = sys.executable   # the hydrolib_env kernel's interpreter (used for workers)
print(f"python  : {PYTHON}")
print(f"script  : {BUILD_SCRIPT}")
print(f"log dir : {LOG_DIR}")
print(f"workers : {MAX_WORKERS}")
print(f"{len(SUBMODELS)} submodels: {SUBMODELS}")

In [ ]:
# ---- build all selected submodels in parallel ----
# Each submodel builds in its own subprocess; MAX_WORKERS run concurrently.
# Per-submodel output is captured to submodels/_build_logs/<name>.log so the
# parallel logs don't interleave.

def _run_one(name):
    sub = SUBMODELS_DIR / name
    log_path = LOG_DIR / f"{name}.log"
    t0 = time.time()
    with open(log_path, "w", encoding="utf-8") as log:
        proc = subprocess.run(
            [PYTHON, str(BUILD_SCRIPT), str(sub)],
            stdout=log, stderr=subprocess.STDOUT, text=True,
        )
    return name, proc.returncode, time.time() - t0, log_path

results = {}
print(f"Building {len(SUBMODELS)} submodels, {MAX_WORKERS} at a time...\n")
t_start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(_run_one, n): n for n in SUBMODELS}
    for fut in as_completed(futures):
        name, rc, dt, log_path = fut.result()
        status = "OK    " if rc == 0 else f"FAILED({rc})"
        print(f"  [{status}] {name}  ({dt/60:.1f} min)")
        results[name] = (rc, dt, str(log_path))

print("\n" + "=" * 70)
print(f"SUMMARY  (wall time {(time.time()-t_start)/60:.1f} min)")
print("=" * 70)
n_ok = 0
for name in SUBMODELS:
    rc, dt, log_path = results.get(name, (None, 0, ""))
    if rc == 0:
        n_ok += 1
        print(f"  {name}: OK        {dt/60:.1f} min")
    else:
        print(f"  {name}: FAILED (exit {rc})  {dt/60:.1f} min  -> {log_path}")
print(f"\n{n_ok}/{len(SUBMODELS)} submodels built successfully.")
if n_ok < len(SUBMODELS):
    print("Inspect the .log files above for failures.")

In [ ]:
# ---- (optional) build or debug a single submodel, live output ----
# Runs one submodel in the foreground and streams its log here. Handy to debug
# a FAILED submodel from the summary above.
name = "submodel_01"
subprocess.run([PYTHON, str(BUILD_SCRIPT), str(SUBMODELS_DIR / name)])